# Exploratory Data Analysis: Ford Used Cars

This notebook explores the Ford subset of the Kaggle **100,000 UK Used Car Data set**. The goal is to understand the dataset before building classical regression models for used car price prediction.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/raw/ford.csv")
PLOTS_DIR = Path("../results/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## Basic Dataset Inspection

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Column names:")
print(df.columns.tolist())

## Missing Values and Duplicate Rows

In [ ]:
missing_values = df.isna().sum()
missing_values

In [ ]:
duplicate_rows = df.duplicated().sum()
print("Duplicate rows:", duplicate_rows)

The raw dataset should be checked for both missing values and duplicate rows before preprocessing. Duplicate rows can bias model training because repeated listings give some observations extra influence.

## Target Variable: Price

In [ ]:
df["price"].describe()

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["price"], bins=40, kde=True)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Number of Cars")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "price_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

Most used car prices are concentrated in the lower and middle price ranges, while a smaller number of expensive cars create a right-skewed distribution. This is common in vehicle price data and should be considered when evaluating regression errors.

## Mileage Distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["mileage"], bins=40, kde=True)
plt.title("Mileage Distribution")
plt.xlabel("Mileage")
plt.ylabel("Number of Cars")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "mileage_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

Mileage is an important predictor because higher mileage usually means more wear. The distribution helps identify whether extreme mileage values may need attention during modeling.

## Year Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="year", order=sorted(df["year"].unique()))
plt.title("Year Distribution")
plt.xlabel("Registration Year")
plt.ylabel("Number of Cars")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "year_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

The year distribution shows which vehicle ages are most represented in the dataset. If most records are from recent years, models may perform best for newer cars and less reliably for very old cars.

## Price Relationships with Numeric Features

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="mileage", y="price", alpha=0.45)
plt.title("Price vs Mileage")
plt.xlabel("Mileage")
plt.ylabel("Price")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "price_vs_mileage.png", dpi=300, bbox_inches="tight")
plt.show()

Price generally decreases as mileage increases, although the relationship is noisy because price also depends on age, model, engine size, fuel type, and transmission.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="year", y="price", alpha=0.45)
plt.title("Price vs Year")
plt.xlabel("Registration Year")
plt.ylabel("Price")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "price_vs_year.png", dpi=300, bbox_inches="tight")
plt.show()

Newer cars tend to have higher prices. This supports creating a `car_age` feature during preprocessing because age is easier to interpret than raw registration year.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="engineSize", y="price", alpha=0.45)
plt.title("Price vs Engine Size")
plt.xlabel("Engine Size")
plt.ylabel("Price")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "price_vs_engine_size.png", dpi=300, bbox_inches="tight")
plt.show()

Engine size can help explain price differences, but it should be interpreted together with model and fuel type because similar engine sizes can appear in very different vehicle classes.

## Average Price by Categorical Features

In [ ]:
avg_price_transmission = df.groupby("transmission", as_index=False)["price"].mean()

plt.figure(figsize=(8, 5))
sns.barplot(data=avg_price_transmission, x="transmission", y="price")
plt.title("Average Price by Transmission")
plt.xlabel("Transmission")
plt.ylabel("Average Price")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "average_price_by_transmission.png", dpi=300, bbox_inches="tight")
plt.show()

Average prices differ across transmission types, so `transmission` should be kept as a categorical feature and encoded before modeling.

In [ ]:
avg_price_fuel = df.groupby("fuelType", as_index=False)["price"].mean()

plt.figure(figsize=(8, 5))
sns.barplot(data=avg_price_fuel, x="fuelType", y="price")
plt.title("Average Price by Fuel Type")
plt.xlabel("Fuel Type")
plt.ylabel("Average Price")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "average_price_by_fuel_type.png", dpi=300, bbox_inches="tight")
plt.show()

Fuel type also affects average price. This makes it useful for classical regression models after categorical encoding.

## Numeric Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include="number")

plt.figure(figsize=(9, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap for Numeric Columns")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

The heatmap gives a quick view of linear relationships between numeric variables. `year`, `mileage`, `engineSize`, `tax`, and `mpg` are useful starting points for regression, while categorical features require encoding first.